In [1]:
%store -r

In [2]:
import commute_dm.core
import commute_dm.ig
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
def make_backend():
    return momapy_kb.lpg.backends.neo4j.Neo4jBackend(
        hostname=credentials.NEO4J_URI,
        username=credentials.NEO4J_USERNAME,
        password=credentials.NEO4J_PASSWORD,
        notifications_min_severity="off",
    )

In [4]:
MAX_LEVELS = [2, 3, 4, 5, 6]
MIN_N_NODES = 5
UPSTREAM_COLLECTION_NAME = commute_dm.core.UPSTREAM_COLLECTION_NAME
DOWNSTREAM_COLLECTION_NAME = commute_dm.core.DOWNSTREAM_COLLECTION_NAME

We compute the interface (between the two activity-flow collections and the AD BEL KG), and build the adjacency index over the stored activity-flow structure. The index is built **once per session** and threaded through the analysis.

In [5]:
with momapy_kb.lpg.session.Session(make_backend()) as session:
    interface = commute_dm.core.get_interface(session)
    index = commute_dm.ig.load_af_index(
        session, [UPSTREAM_COLLECTION_NAME, DOWNSTREAM_COLLECTION_NAME]
    )
len(interface), len(index.species), len(index.modulation_endpoints), len(index.gates)

(127, 8247, 9603, 40)

In [6]:
commute_dm.utils.remake_dir(INTERFACE_ANALYSIS_GRAPHS_DIR)

We select and render the sub-maps upstream of the COVID seeds and downstream of the PD seeds. Every element in the output is a **stored** activity-flow element (species, signed modulation, boolean logic gate, glyph, arc), except the synthetic central node standing for the interface protein itself.

In [7]:
with momapy_kb.lpg.session.Session(make_backend()) as session:
    stats_df = commute_dm.core.make_and_write_cd_maps_from_interface(
        session=session,
        interface=interface,
        index=index,
        output_dir_path=INTERFACE_ANALYSIS_GRAPHS_DIR,
        upstream_collection_name=UPSTREAM_COLLECTION_NAME,
        downstream_collection_name=DOWNSTREAM_COLLECTION_NAME,
        max_levels=MAX_LEVELS,
        upstream_nodes_color="lightblue",
        downstream_nodes_color="lightgreen",
        common_nodes_color="goldenrod",
        interface_nodes_color="red",
        min_n_nodes=MIN_N_NODES,
        include_compartment_layouts=True,
    )
stats_df

,identifier,display_name,max_level,n_species,n_modulations,n_gates,n_compartments,n_compartment_layouts,n_templates,n_renumbered,n_elements_without_glyph,n_modulations_dropped_no_glyph,n_extra_influences_dropped,n_subunit_entries,n_interned_away,n_colored
0,P30556,AGTR1,2,72,110,0,17,16,34,581,2,0,2,18,0,72
1,P30556,AGTR1,3,119,189,0,21,20,55,998,2,0,2,66,0,119
2,P30556,AGTR1,4,166,263,0,22,21,75,1405,2,0,2,110,0,166
3,P30556,AGTR1,5,234,358,0,27,26,115,2018,2,0,2,165,0,234
4,P30556,AGTR1,6,310,493,0,32,31,159,2778,2,0,2,237,0,310
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
227,Q96P20,NLRP3,6,165,232,4,14,13,95,1432,3,0,4,158,0,165
228,P02787,TF,3,18,23,0,13,12,4,243,1,0,2,92,0,18
229,P02787,TF,4,23,30,0,17,16,4,307,1,0,2,114,0,23
230,P02787,TF,5,29,41,0,18,17,4,362,1,0,2,126,0,29
